In [1]:
import json 

real_path = '../synthetic_sampling/samples/total_real_data.json'

with open(real_path, 'r') as f:
    real_data = json.load(f)

filename_fewshot = "saves/fewshot_responses_final.json"
filename_zeroshot = "saves/zeroshot_responses_final.json"

with open(filename_fewshot, 'r') as f:
    fewshot = json.load(f)

with open(filename_zeroshot, 'r') as f:
    zeroshot = json.load(f)

In [2]:
import re
from itertools import permutations
from collections import defaultdict
import numpy as np

# -------------------------
# Span matching helpers
# -------------------------
_TOKEN_RE = re.compile(r"[A-Za-z0-9]+", re.UNICODE)

def _norm(s: str) -> str:
    return " ".join(str(s).strip().strip('"').strip("'").split())

def _tokenize(s: str):
    return set(_TOKEN_RE.findall(_norm(s).lower()))

def _iou(a: str, b: str) -> float:
    ta, tb = _tokenize(a), _tokenize(b)
    if not ta and not tb:
        return 1.0
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

def match_spans(gold: list, pred: list, iou_threshold: float = 0.5):
    """Greedy 1-1 IoU matching. Returns (tp, fp, fn)."""
    gold = [_norm(s) for s in gold if _norm(s)]
    pred = [_norm(s) for s in pred if _norm(s)]

    if not gold and not pred:
        return 0, 0, 0
    if not gold:
        return 0, len(pred), 0
    if not pred:
        return 0, 0, len(gold)

    candidates = []
    for gi, g in enumerate(gold):
        for pi, p in enumerate(pred):
            iou = _iou(g, p)
            if iou >= iou_threshold:
                candidates.append((iou, gi, pi))

    candidates.sort(reverse=True, key=lambda x: x[0])
    matched_g, matched_p = set(), set()
    tp = 0
    for iou, gi, pi in candidates:
        if gi in matched_g or pi in matched_p:
            continue
        matched_g.add(gi)
        matched_p.add(pi)
        tp += 1

    return tp, len(pred) - tp, len(gold) - tp

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}

def pairwise_f1(gold_per_doc: list, pred_per_doc: list, iou_threshold: float = 0.5):
    n = min(len(gold_per_doc), len(pred_per_doc))
    ann = {"gold": gold_per_doc[:n], "pred": pred_per_doc[:n]}

    total_tp = total_fp = total_fn = 0
    per_pair = {}

    macro_precisions = []
    macro_recalls = []
    macro_f1s = []

    for a, b in permutations(["gold", "pred"], 2):
        tp = fp = fn = 0
        per_doc_scores = []

        for i in range(n):
            tpi, fpi, fni = match_spans(ann[a][i], ann[b][i], iou_threshold)

            tp += tpi
            fp += fpi
            fn += fni

            doc_scores = _prf(tpi, fpi, fni)
            per_doc_scores.append(doc_scores)

        per_pair[f"{a}→{b}"] = _prf(tp, fp, fn)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        macro_precisions.append(np.mean([d["precision"] for d in per_doc_scores]) if per_doc_scores else 0.0)
        macro_recalls.append(np.mean([d["recall"] for d in per_doc_scores]) if per_doc_scores else 0.0)
        macro_f1s.append(np.mean([d["f1"] for d in per_doc_scores]) if per_doc_scores else 0.0)

    return {
        "pairwise_micro": _prf(total_tp, total_fp, total_fn),
        "pairwise_macro": {
            "precision": round(float(np.mean(macro_precisions)) if macro_precisions else 0.0, 4),
            "recall": round(float(np.mean(macro_recalls)) if macro_recalls else 0.0, 4),
            "f1": round(float(np.mean(macro_f1s)) if macro_f1s else 0.0, 4),
        },
        "per_pair": per_pair,
        "num_docs": n,
        "iou_threshold": iou_threshold,
    }

def token_level_kappa(gold_per_doc: list, pred_per_doc: list, documents: list):
    """
    Token-level Cohen's Kappa.
    For each document, project gold and pred spans onto whitespace-tokenized
    document tokens as binary vectors, then compute kappa across all tokens.
    """
    all_gold, all_pred = [], []

    for doc_text, gold_spans, pred_spans in zip(documents, gold_per_doc, pred_per_doc):
        tokens = doc_text.split()
        n_tok = len(tokens)

        def project(spans):
            labels = [0] * n_tok
            for span in spans:
                span_toks = span.split()
                slen = len(span_toks)
                for i in range(n_tok - slen + 1):
                    if tokens[i:i + slen] == span_toks:
                        for j in range(i, i + slen):
                            labels[j] = 1
                        break
            return labels

        all_gold.extend(project(gold_spans))
        all_pred.extend(project(pred_spans))

    y_g = np.array(all_gold, dtype=int)
    y_p = np.array(all_pred, dtype=int)

    n = len(y_g)
    if n == 0:
        return {"kappa": 0.0, "num_tokens": 0}

    p_o = float((y_g == y_p).mean())
    p_g1 = float(y_g.mean())
    p_p1 = float(y_p.mean())
    p_e = (p_g1 * p_p1) + ((1 - p_g1) * (1 - p_p1))
    kappa = (p_o - p_e) / (1 - p_e) if abs(1 - p_e) > 1e-12 else 0.0

    tp = int(((y_g == 1) & (y_p == 1)).sum())
    fp = int(((y_g == 0) & (y_p == 1)).sum())
    fn = int(((y_g == 1) & (y_p == 0)).sum())

    return {
        "kappa": round(kappa, 4),
        "token_precision": round(tp / (tp + fp) if (tp + fp) > 0 else 0.0, 4),
        "token_recall": round(tp / (tp + fn) if (tp + fn) > 0 else 0.0, 4),
        "token_f1": round(2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else 0.0, 4),
        "num_tokens": n,
        "debug": {"p_o": round(p_o, 4), "p_e": round(p_e, 4), "tp": tp, "fp": fp, "fn": fn},
    }


# -------------------------
# Group pred data by category
# -------------------------
LABEL_MAP = {
    "unsupported_claim": "Unsupported claim",
    "unsupported claim": "Unsupported claim",
    "format": "Format",
    "coherence": "Coherence",
    "lacks_synthesis": "Lacks synthesis",
    "lacks synthesis": "Lacks synthesis",
}

def group_pred_by_category(pred_data: list) -> dict:
    grouped = defaultdict(list)

    for doc in pred_data:
        document = doc.get("document", "")
        categories = doc.get("categories", {})

        for raw_cat, spans_list in categories.items():
            cat = LABEL_MAP.get(raw_cat.strip().lower())
            if cat is None:
                continue

            spans = [
                s["span_text"] for s in spans_list
                if s.get("span_text") is not None
            ]

            if spans:
                grouped[cat].append({"document": document, "spans": spans})

    return dict(grouped)


# -------------------------
# Match docs between real and pred
# -------------------------
def align_by_document(real_docs: list, pred_docs: list):
    """
    Match real and pred entries by document text.
    Returns aligned (gold_spans, pred_spans, document) triples.
    Only includes documents present in both.
    """
    pred_index = {}
    for d in pred_docs:
        key = d["document"].strip()[:200]  # use first 200 chars as key
        pred_index[key] = d

    aligned_gold, aligned_pred, aligned_docs = [], [], []
    unmatched = 0

    for real_doc in real_docs:
        key = real_doc["document"].strip()[:200]
        if key not in pred_index:
            unmatched += 1
            continue

        gold_spans = [s["span"] for s in real_doc.get("spans", []) if s.get("span")]
        pred_spans = pred_index[key].get("spans", [])

        aligned_gold.append(gold_spans)
        aligned_pred.append(pred_spans)
        aligned_docs.append(real_doc["document"])

    if unmatched > 0:
        print(f"  [WARNING] {unmatched} real docs had no matching pred doc")

    return aligned_gold, aligned_pred, aligned_docs


# -------------------------
# Main evaluation
# -------------------------
def evaluate_gpt_vs_real(
    real_data: dict,
    pred_data: list,
    iou_threshold: float = 0.5,
    label: str = "fewshot",
):
    pred_grouped = group_pred_by_category(pred_data)
    results = {}
    
    for category, real_docs in real_data.items():
        pred_docs = pred_grouped.get(category, [])

        print(f"\n[{label}] Category: {category}")
        print(f"  Real docs: {len(real_docs)}, Pred docs: {len(pred_docs)}")

        if not pred_docs:
            print(f"  No predictions found for {category} — skipping")
            continue

        gold_spans, pred_spans, documents = align_by_document(real_docs, pred_docs)

        print(f"  Matched docs: {len(gold_spans)}")

        pf1 = pairwise_f1(gold_spans, pred_spans, iou_threshold=iou_threshold)
        kappa = token_level_kappa(gold_spans, pred_spans, documents)

        results[category] = {
            "pairwise_f1": pf1,
            "token_kappa": kappa,
        }

        print(f"  Pairwise micro F1: {pf1['pairwise_micro']}")
        print(f"  Token kappa: {kappa['kappa']}")

    return results

In [ ]:
iou_threshold = 0.5

print("=" * 50)
print("FEWSHOT vs REAL")
print("=" * 50)
fewshot_results = evaluate_gpt_vs_real(
    real_data=real_data,
    pred_data=fewshot,
    iou_threshold=iou_threshold,
    label="fewshot",
)

print("\n" + "=" * 50)
print("ZEROSHOT vs REAL")
print("=" * 50)
zeroshot_results = evaluate_gpt_vs_real(
    real_data=real_data,
    pred_data=zeroshot,
    iou_threshold=iou_threshold,
    label="zeroshot",
)

FEWSHOT vs REAL

[fewshot] Category: Unsupported claim
  Real docs: 54, Pred docs: 89
  Matched docs: 54
  Pairwise micro F1: {'precision': 0.1132, 'recall': 0.1132, 'f1': 0.1132}
  Token kappa: 0.0682

[fewshot] Category: Format
  Real docs: 22, Pred docs: 69
  Matched docs: 22
  Pairwise micro F1: {'precision': 0.3304, 'recall': 0.3304, 'f1': 0.3304}
  Token kappa: 0.2173

[fewshot] Category: Coherence
  Real docs: 19, Pred docs: 86
  Matched docs: 19
  Pairwise micro F1: {'precision': 0.2992, 'recall': 0.2992, 'f1': 0.2992}
  Token kappa: 0.1038

[fewshot] Category: Lacks synthesis
  Real docs: 22, Pred docs: 89
  Matched docs: 22
  Pairwise micro F1: {'precision': 0.044, 'recall': 0.044, 'f1': 0.044}
  Token kappa: 0.0655

ZEROSHOT vs REAL

[zeroshot] Category: Unsupported claim
  Real docs: 54, Pred docs: 89
  Matched docs: 54
  Pairwise micro F1: {'precision': 0.1065, 'recall': 0.1065, 'f1': 0.1065}
  Token kappa: 0.0326

[zeroshot] Category: Format
  Real docs: 22, Pred docs: 65

In [3]:
def extract_spans_from_bio(tokens, labels, target_label="UNSUPPORTED_CLAIM"):
    """Extract span texts from BIO-tagged tokens."""
    spans = []
    current = []

    for token, label in zip(tokens, labels):
        if label == f"B-{target_label}":
            if current:
                spans.append(" ".join(current))
            current = [token]
        elif label == f"I-{target_label}" and current:
            current.append(token)
        else:
            if current:
                spans.append(" ".join(current))
            current = []

    if current:
        spans.append(" ".join(current))

    return spans

def _make_key(text: str) -> str:
    return " ".join(text.strip().lower().split())[:200]

def extract_spans_from_category_entry(entry, category="Unsupported claim"):
    """Extract span texts and their segment texts from the prediction format."""
    results = []
    for item in entry.get("categories", {}).get(category, []):
        if item.get("gpt_label") == category and item.get("span_text"):
            results.append({
                "span_text": item["span_text"].strip(),
                "segment_text": (item.get("segment_text") or "").strip(),
            })
    return results

def evaluate_ner_vs_pred_by_length(ner_data, pred_data, threshold=5, iou_threshold=0.5, mode="both"):
    """
    mode: "short" | "long" | "both"
    """
    category = "Unsupported claim"

    pred_by_segment = defaultdict(list)
    all_pred_segments = set()
    for entry in pred_data:
        for item in entry.get("categories", {}).get(category, []):
            seg_key = _make_key(item.get("segment_text", ""))
            if seg_key:
                all_pred_segments.add(seg_key)
            if item.get("gpt_label") == category and item.get("span_text"):
                pred_by_segment[seg_key].append(item["span_text"].strip())

    short_tp = short_fp = short_fn = 0
    long_tp  = long_fp  = long_fn  = 0

    all_short_gold, all_short_pred, all_short_docs = [], [], []
    all_long_gold,  all_long_pred,  all_long_docs  = [], [], []

    matched = unmatched = 0

    for entry in ner_data:
        sentence = " ".join(entry["tokens"])
        seg_key = _make_key(sentence)

        if seg_key not in all_pred_segments:
            unmatched += 1
            continue

        matched += 1

        gold_spans = extract_spans_from_bio(
            entry["tokens"], entry["labels"], target_label="UNSUPPORTED_CLAIM"
        )
        matched_pred_spans = pred_by_segment.get(seg_key, [])

        short_gold = [s for s in gold_spans if len(s.split()) < threshold]
        long_gold  = [s for s in gold_spans if len(s.split()) >= threshold]
        short_pred = [s for s in matched_pred_spans if len(s.split()) < threshold]
        long_pred  = [s for s in matched_pred_spans if len(s.split()) >= threshold]

        all_short_gold.append(short_gold)
        all_short_pred.append(short_pred)
        all_short_docs.append(sentence)

        all_long_gold.append(long_gold)
        all_long_pred.append(long_pred)
        all_long_docs.append(sentence)

        # Token-level TP/FP/FN via projection onto sentence tokens
        tokens = entry["tokens"]
        n_tok = len(tokens)

        def project_spans(spans):
            labels = [0] * n_tok
            for span in spans:
                span_toks = span.split()
                slen = len(span_toks)
                for i in range(n_tok - slen + 1):
                    if tokens[i:i + slen] == span_toks:
                        for j in range(i, i + slen):
                            labels[j] = 1
                        break
            return labels

        short_gold_tok = project_spans(short_gold)
        short_pred_tok = project_spans(short_pred)
        long_gold_tok  = project_spans(long_gold)
        long_pred_tok  = project_spans(long_pred)

        short_tp += sum(1 for g, p in zip(short_gold_tok, short_pred_tok) if g == 1 and p == 1)
        short_fp += sum(1 for g, p in zip(short_gold_tok, short_pred_tok) if g == 0 and p == 1)
        short_fn += sum(1 for g, p in zip(short_gold_tok, short_pred_tok) if g == 1 and p == 0)

        long_tp += sum(1 for g, p in zip(long_gold_tok, long_pred_tok) if g == 1 and p == 1)
        long_fp += sum(1 for g, p in zip(long_gold_tok, long_pred_tok) if g == 0 and p == 1)
        long_fn += sum(1 for g, p in zip(long_gold_tok, long_pred_tok) if g == 1 and p == 0)

    print(f"  Matched NER sentences: {matched}, Skipped (not in pred): {unmatched}")

    short_metrics = _prf(short_tp, short_fp, short_fn)
    long_metrics  = _prf(long_tp,  long_fp,  long_fn)

    # Pairwise F1
    short_pf1 = pairwise_f1(all_short_gold, all_short_pred, iou_threshold=iou_threshold)
    long_pf1  = pairwise_f1(all_long_gold,  all_long_pred,  iou_threshold=iou_threshold)

    # Cohen's kappa
    short_kappa = token_level_kappa(all_short_gold, all_short_pred, all_short_docs)
    long_kappa  = token_level_kappa(all_long_gold,  all_long_pred,  all_long_docs)

    if mode in ("short", "both"):
        print(f"\n=== Unsupported Claim: Short spans (< {threshold} words) ===")
        print(f"  TP: {short_tp}, FP: {short_fp}, FN: {short_fn}")
        print(f"  Precision: {short_metrics['precision']}, Recall: {short_metrics['recall']}, F1: {short_metrics['f1']}")
        print(f"  Pairwise F1: {short_pf1['pairwise_micro']}")
        print(f"  Token Kappa: {short_kappa['kappa']}")

    if mode in ("long", "both"):
        print(f"\n=== Unsupported Claim: Long spans (>= {threshold} words) ===")
        print(f"  TP: {long_tp}, FP: {long_fp}, FN: {long_fn}")
        print(f"  Precision: {long_metrics['precision']}, Recall: {long_metrics['recall']}, F1: {long_metrics['f1']}")
        print(f"  Pairwise F1: {long_pf1['pairwise_micro']}")
        print(f"  Token Kappa: {long_kappa['kappa']}")

    return {
        "short": {
            **short_metrics,
            "tp": short_tp, "fp": short_fp, "fn": short_fn,
            "pairwise_f1": short_pf1,
            "kappa": short_kappa["kappa"],
            "token_kappa_debug": short_kappa.get("debug", {}),
            "gold_spans": [s for spans in all_short_gold for s in spans],
            "pred_spans": [s for spans in all_short_pred for s in spans],
        },
        "long": {
            **long_metrics,
            "tp": long_tp, "fp": long_fp, "fn": long_fn,
            "pairwise_f1": long_pf1,
            "kappa": long_kappa["kappa"],
            "token_kappa_debug": long_kappa.get("debug", {}),
            "gold_spans": [s for spans in all_long_gold for s in spans],
            "pred_spans": [s for spans in all_long_pred for s in spans],
        },
    }

In [4]:
import json
import os

with open("../training/data/unsupported_claim/unsupported_claim_ner_eval_short_1_5w.json", 'r') as f:
    ner_data_short = json.load(f)

with open("../training/data/unsupported_claim/unsupported_claim_ner_eval_long_6plus.json", 'r') as f:
    ner_data_long = json.load(f)

print("=" * 50)
print("FEWSHOT vs REAL")
print("=" * 50)
fewshot_results = evaluate_gpt_vs_real(real_data, fewshot, label="fewshot")
fewshot_short = evaluate_ner_vs_pred_by_length(ner_data_short, fewshot, threshold=5, mode="short")
fewshot_long  = evaluate_ner_vs_pred_by_length(ner_data_long,  fewshot, threshold=6, mode="long")

print("\n" + "=" * 50)
print("ZEROSHOT vs REAL")
print("=" * 50)
zeroshot_results = evaluate_gpt_vs_real(real_data, zeroshot, label="zeroshot")
zeroshot_short = evaluate_ner_vs_pred_by_length(ner_data_short, zeroshot, threshold=5, mode="short")
zeroshot_long  = evaluate_ner_vs_pred_by_length(ner_data_long,  zeroshot, threshold=6, mode="long")


fewshot_results["Unsupported claim (short)"] = fewshot_short["short"]
fewshot_results["Unsupported claim (long)"]  = fewshot_long["long"]
fewshot_results.pop("Unsupported claim", None)

zeroshot_results["Unsupported claim (short)"] = zeroshot_short["short"]
zeroshot_results["Unsupported claim (long)"]  = zeroshot_long["long"]
zeroshot_results.pop("Unsupported claim", None)

with open("saves/results_fewshot.json", 'w') as f:
    json.dump(fewshot_results, f, indent=2)

with open("saves/results_zeroshot.json", 'w') as f:
    json.dump(zeroshot_results, f, indent=2)


FEWSHOT vs REAL

[fewshot] Category: Unsupported claim
  Real docs: 54, Pred docs: 88
  [WARNING] 1 real docs had no matching pred doc
  Matched docs: 53
  Pairwise micro F1: {'precision': 0.1114, 'recall': 0.1114, 'f1': 0.1114}
  Token kappa: 0.0535

[fewshot] Category: Format
  Real docs: 22, Pred docs: 66
  Matched docs: 22
  Pairwise micro F1: {'precision': 0.3922, 'recall': 0.3922, 'f1': 0.3922}
  Token kappa: 0.1565

[fewshot] Category: Coherence
  Real docs: 19, Pred docs: 86
  Matched docs: 19
  Pairwise micro F1: {'precision': 0.2783, 'recall': 0.2783, 'f1': 0.2783}
  Token kappa: 0.1288

[fewshot] Category: Lacks synthesis
  Real docs: 22, Pred docs: 89
  Matched docs: 22
  Pairwise micro F1: {'precision': 0.0433, 'recall': 0.0433, 'f1': 0.0433}
  Token kappa: 0.0405
  Matched NER sentences: 26, Skipped (not in pred): 0

=== Unsupported Claim: Short spans (< 5 words) ===
  TP: 18, FP: 10, FN: 48
  Precision: 0.6429, Recall: 0.2727, F1: 0.383
  Pairwise F1: {'precision': 0.4, 